In this notebook, we visualize several samples featuring different jammer types and demonstrate how the VAE detects the jammer in each sample. 

## Import

In [ ]:
import torch

import numpy as np
import matplotlib.pyplot as plt
import sys
import os

COL_WIDTH = 3.5 # in inches
TWO_COL_WIDTH = 7.16 # in inches
RATIO = 1.618 # golden ratio
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 8

In [ ]:
sys.path.append("../")

from utils.functions import set_random, load_config, low_pass_filter_fft, max_cosine_similarities_plot
from src.models import Encoder, Decoder
from src.data_loading import load_labels, SpectralImagesSample, get_dataset_path
from src.train_unsupervised import base_transform

set_random(42)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device = ', device)

# Data and Model Loading

Here determine the path for the dataset

In [ ]:
config = load_config(os.path.join("..", "src", "config.yml"))
data_path = get_dataset_path(config["dataset_nr"])
df = load_labels(data_path)

Load the learned weights for VAE

In [ ]:
encoder = Encoder()
decoder = Decoder()
    
encoder.load_state_dict(
    torch.load(os.path.join("..", "Weights", "OUT", "VAE_1027-220937", "encoder_final.pt"), map_location = device, weights_only=True),
    strict=False
    )
decoder.load_state_dict(
    torch.load(os.path.join("..", "Weights", "OUT", "VAE_1027-220937", "decoder_final.pt"), map_location = device, weights_only=True),
    strict=False
    )

In [ ]:
encoder.to(device)
encoder.eval()

In [ ]:
decoder.to(device)
decoder.eval()

# Visualisation

In [ ]:
# -------------------------------------------
# Configuration
# -------------------------------------------
 
N_ROWS = 21
COL_KEYS = ["image", "decoded", "loss", "f_loss"]
COL_TITLES = ["Original Image", "VAE Reconstruction", "Reconstruction Loss", "Filtered Loss"]
LOSS_VMIN, LOSS_VMAX = -0.001, 0.1
LOSS_TICKS = [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]
LPF_CUTOFF = 15

LOSS_COLS = {2, 3}   
FILTER_COL = 3      

# -------------------------------------------
# Helper Functions
# -------------------------------------------

def compute_loss(original: np.ndarray, reconstructed: np.ndarray) -> np.ndarray:
    """Clipped, element-wise reconstruction loss."""
    return np.clip((original - reconstructed) * original, 0, None)

def add_colorbar(fig, ax, im) -> None:
    """Attach a fixed colorbar to axes."""
    cbar = fig.colorbar(im, ax=ax, fraction=0.07, pad=0.04)
    cbar.ax.tick_params(labelsize=12)
    cbar.set_ticks(LOSS_TICKS)

## Normal Sample

In [ ]:
df[df['name']==9020]

In [ ]:
normal_sampel = SpectralImagesSample(
    sample_id=9020,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,
    )

In [ ]:
n_dict = {}
n_dict['image'] = np.array(normal_sampel.frames)

imgs = torch.stack(normal_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = torch.zeros_like(mu)
for i in range(100):
    epsilon = torch.randn_like(std).to(device)
    z += mu + std * epsilon
z /= 100
decoded = decoder(z)

n_dict['decoded'] = decoded.detach().cpu().numpy()
n_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = n_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")

        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()

In [ ]:
similarity_scores_mu = max_cosine_similarities_plot(np.array(data["latent"]))

## Barrage Jammer

In [ ]:
df[df['name']==15025]

In [ ]:
barrage_sampel = SpectralImagesSample(
    sample_id=15025,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,
    )

In [ ]:
b_dict = {}
b_dict['image'] = np.array(barrage_sampel.frames)

imgs = torch.stack(barrage_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = torch.zeros_like(mu)
for i in range(100):
    epsilon = torch.randn_like(std).to(device)
    z += mu + std * epsilon
z /= 100
decoded = decoder(z)


b_dict['decoded'] = decoded.detach().cpu().numpy()
b_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = b_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()

In [ ]:
similarity_scores_mu = max_cosine_similarities_plot(np.array(data["latent"]))

## Deceptive Jammer

In [ ]:
df[df['name'] == 11009]

In [ ]:
deceptive_sampel = SpectralImagesSample(
    sample_id=11009,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,
    )

In [ ]:
d_dict = {}
d_dict['image'] = np.array(deceptive_sampel.frames)

imgs = torch.stack(deceptive_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = torch.zeros_like(mu)
for i in range(100):
    epsilon = torch.randn_like(std).to(device)
    z += mu + std * epsilon
z /= 100
decoded = decoder(z)

d_dict['decoded'] = decoded.detach().cpu().numpy()
d_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = d_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")

        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()

## Pilot Jammer

In [ ]:
df[df['name'] == 17001]

In [ ]:
Pilot_sampel = SpectralImagesSample(
    sample_id=17001,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,    
    )

In [ ]:
p_dict = {}
p_dict['image'] = np.array(Pilot_sampel.frames)

imgs = torch.stack(Pilot_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = mu
decoded = decoder(z)


p_dict['decoded'] = decoded.detach().cpu().numpy()
p_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = p_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(6.5, 9))
loss_list: list[float] = []

r = 0
l_img = 0
for col, key in enumerate(COL_KEYS):
    
    for row in range(N_ROWS):
        
        if row not in (15, 17, 20):
            continue
        
        if col >2:
            continue

        ax = axes[r, col]
        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                      # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            l_img = im


        #ax.set_title(f"SU: {row}")
        ax.axis("off")
        r+=1
    r = 0

# [left, bottom, width, height] in figure coordinates (0–1)
cbar_ax = fig.add_axes([0.99, 0.05, 0.02, 0.9])

cbar = fig.colorbar(im, cax=cbar_ax, orientation="vertical")

for label in cbar.ax.get_yticklabels():
    label.set_fontweight("bold")

cbar.ax.tick_params(labelsize=17)
#for col, title in enumerate(COL_TITLES[:-1]):
#    axes[0, col].set_title(f"{title}\nSU 15")

plt.tight_layout()
plt.show()

In [ ]:
data = p_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")

        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()

## Sweep Jammer

In [ ]:
df[df['name']==13002]

In [ ]:
sweep_sampel = SpectralImagesSample(
    sample_id=13002,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,
    )

In [ ]:
s_dict = {}
s_dict['image'] = np.array(sweep_sampel.frames)

imgs = torch.stack(sweep_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = torch.zeros_like(mu)
for i in range(100):
    epsilon = torch.randn_like(std).to(device)
    z += mu + std * epsilon
z /= 100
decoded = decoder(z)


s_dict['decoded'] = decoded.detach().cpu().numpy()
s_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = s_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")

        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()

## Random Hopping Jammer

In [ ]:
df[df['name'] == 19006]

In [ ]:
RH_sampel = SpectralImagesSample(
    sample_id=19006,
    path=data_path,
    transform=base_transform,
    dataframe=df,
    max=14.244341456062713, 
    min=-91.57541304998105,  
    )

In [ ]:
r_dict = {}
r_dict['image'] = np.array(RH_sampel.frames)

imgs = torch.stack(RH_sampel.frames).float()
imgs = imgs.to(device)

mu, log_var = encoder(imgs)
std = torch.exp(0.5 * log_var)
z = torch.zeros_like(mu)
for i in range(100):
    epsilon = torch.randn_like(std).to(device)
    z += mu + std * epsilon
z /= 100
decoded = decoder(z)


r_dict['decoded'] = decoded.detach().cpu().numpy()
r_dict['latent'] = mu.detach().cpu().numpy()

In [ ]:
data = r_dict
loss_images = compute_loss(data["image"], data["decoded"])   # shape (N, 1, H, W)

# -------------------------------------------
# Plot
# -------------------------------------------
fig, axes = plt.subplots(nrows=N_ROWS, ncols=len(COL_KEYS), figsize=(8.5, 65))
loss_list: list[float] = []

for col, key in enumerate(COL_KEYS):
    for row in range(N_ROWS):
        ax = axes[row, col]

        if col not in LOSS_COLS:                        # raw image or reconstruction
            ax.imshow(data[key][row, 0], cmap="gray")

        elif col != FILTER_COL:                         # unfiltered loss
            im = ax.imshow(loss_images[row, 0], vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")

        else:                                           # filtered loss
            filtered = low_pass_filter_fft(loss_images[row, 0], LPF_CUTOFF)
            im = ax.imshow(filtered, vmin=LOSS_VMIN, vmax=LOSS_VMAX, cmap="viridis")
            loss_list.append(float(np.max(filtered)))
            add_colorbar(fig, ax, im)

        ax.set_title(f"su: {row}")
        ax.axis("off")

for col, title in enumerate(COL_TITLES):
    axes[0, col].set_title(f"{title}\nsu: 0")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.arange(21), np.array(loss_list), marker='o', linestyle='--')
plt.xticks(range(21))
plt.xlabel("SUs")
plt.ylabel("Rec Loss")
plt.title("Reconstruction Loss for each SU")
plt.grid(True)
plt.show()